# Stage 1 enrichment on Kaggle (2 x T4)

Runs `scripts/enrich_corpus.py` on Kaggle's two free GPUs, one Ollama server and one enrichment process per GPU, each on its own half of the documents.

**Read `docs/kaggle-run.md` first** - it lists what has to be set up by hand (dataset, internet, GPU, secret) and how to continue over several sessions.

- **Only edit the first code cell.** Everything else reads its settings from there.
- `MODE = "sample"` enriches the 40-document sample on one GPU (about 15 minutes) and compares it with the Mac's file. **Do this once before the full run.**
- `MODE = "full"` runs the whole corpus. Use **Save Version -> Save & Run All (Commit)**, not the interactive editor: the files in `/kaggle/working` are only kept when a committed run *finishes*, which is why each process stops itself 45 minutes before the session limit.

Policy note: this is still the project's open-weight model (`qwen2.5:14b`), with the same weights, prompt, seed and settings as on the Mac. Only the computer is borrowed. Nothing is sent to a paid API. See `docs/06_POLICY_EXCEPTIONS.md`.

In [ ]:
# ============================== SETTINGS - the only cell to edit ==============================
import time
SESSION_STARTED = time.time()      # the session clock; the time limit below is counted from here

MODE = "sample"                    # "sample" = 40 documents on one GPU;  "full" = the whole corpus on two

REPO_URL = "https://github.com/fypsvvy-boss/Structure-Grounded-SIRA.git"
COMMIT = ""                        # REQUIRED: the full 40-character commit hash to run (git rev-parse HEAD)
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"   # NAME of the Kaggle secret holding a GitHub token (private repo only)

MODEL = "qwen2.5:14b"
# What the Mac holds (curl -s localhost:11434/api/tags). `ollama list` shows the first 12 characters.
EXPECTED_DIGEST = "7cdf5a0187d5c58cc5d369b255592f7841d1c4696d45a8c8a9489440385b22f6"
EXPECTED_CONFIG_HASH = "364d30da6c75"
OLLAMA_VERSION = "0.33.2"          # the Mac's release, so both machines run the same engine
PYSERINI_VERSION = "2.3.0"         # the Mac's; it reads the base index for the too_common gate

SESSION_LIMIT_HOURS = 12.0         # Kaggle's limit for one session
SAFETY_MINUTES = 45                # each process stops this long before the limit
PROGRESS_EVERY_MINUTES = 10

# Full run: one entry per GPU. Both take the catalogues first and CVEs last (as planned for the Mac),
# and --shard splits every kind in half by a hash of the document id, so the two never overlap.
RUN_NAME = "corpus_full_v4_14b"
PLAN = [
    {"gpu": 0, "port": 11434, "kinds": "cwe,capec,attack,cve", "shard": "1/2"},
    {"gpu": 1, "port": 11435, "kinds": "cwe,capec,attack,cve", "shard": "2/2"},
]
FULL_ARGS = ["--model", MODEL, "--name-scorer", "v2"]      # the Stage 1 command in module1-freeze.md

# Sample run: exactly the flags that made the Mac's reference file.
SAMPLE_ARGS = ["--per-kind", "10", "--model", MODEL]
SAMPLE_OUTPUT = "corpus_stratified_v4_14b_sections-v2_kaggle.jsonl"
MAC_REFERENCE = "corpus_stratified_v4_14b_sections-v2.jsonl"   # put it in the dataset to compare here

# The uploaded base index must answer like the Mac's (indexes/base).
EXPECTED_TOTAL_DOCS = 6044
EXPECTED_DOC_FREQS = {"attack": 3989}

INPUT_ROOT = "/kaggle/input"
WORKING = "/kaggle/working"
REPO_DIR = "/tmp/sira-repo"        # outside /kaggle/working, so the checkout is not saved as output

assert MODE in ("sample", "full"), MODE
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT), \
    "Set COMMIT to the full 40-character hash of the commit to run."
print(f"mode={MODE}  commit={COMMIT[:12]}  model={MODEL}")

## 1. The GPUs

In [ ]:
import os, subprocess, sys

def sh(cmd, check=True, quiet=False, **kwargs):
    """Run a command (a shell string, or a list of arguments); print what it printed."""
    result = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kwargs)
    if not quiet and (result.stdout + result.stderr).strip():
        print((result.stdout + result.stderr).strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed ({result.returncode}): {cmd}")
    return result

gpus = sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader").stdout.strip().splitlines()
needed = 2 if MODE == "full" else 1
assert len(gpus) >= needed, (
    f"{len(gpus)} GPU(s) visible, {needed} needed. Settings -> Accelerator -> GPU T4 x2, then run again.")

## 2. Install Ollama, Java and Pyserini

In [ ]:
# zstd: the Ollama installer needs it to unpack. Java 21: Pyserini's Lucene needs it.
sh("apt-get update -qq && apt-get install -y -qq zstd openjdk-21-jdk-headless > /dev/null", quiet=True)
java_homes = sorted(p for p in os.listdir("/usr/lib/jvm") if p.startswith("java-21"))
assert java_homes, "Java 21 did not install"
os.environ["JAVA_HOME"] = f"/usr/lib/jvm/{java_homes[0]}"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]
sh("java -version")

sh(f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh > /tmp/ollama-install.log 2>&1")
sh("ollama --version", check=False)        # the client prints a warning while no server is running; that is fine

sh([sys.executable, "-m", "pip", "install", "-q", f"pyserini=={PYSERINI_VERSION}", "networkx", "PyYAML"])
import importlib.metadata as md
print({name: md.version(name) for name in ("pyserini", "pyjnius", "networkx", "PyYAML")})

## 3. The code, at the commit named above

A full clone (not a shallow one): the manifest records the last commit that touched `src/`, `scripts/` or `configs/`, and a shallow clone cannot see that history.

The GitHub token, if the repository is private, comes from Kaggle Secrets. It is passed to git through the environment for this one command: it is never written into the notebook, the clone's config, or the output.

In [ ]:
import base64, shutil

def github_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
    except Exception:
        return None

def clone(token):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraheader",
                    "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}"})
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    return subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], env=env, capture_output=True, text=True)

result = clone(None)
if result.returncode != 0:
    token = github_token()
    assert token, (f"Could not clone {REPO_URL} without a token, and no Kaggle secret named "
                   f"{GITHUB_TOKEN_SECRET!r} is attached (Add-ons -> Secrets).")
    result = clone(token)
    del token
    assert result.returncode == 0, "git clone failed even with the token (wrong repository, or the token cannot read it)."

sh(f"git -C {REPO_DIR} checkout --quiet --detach {COMMIT}")
head = sh(f"git -C {REPO_DIR} rev-parse HEAD", quiet=True).stdout.strip()
assert head == COMMIT, (head, COMMIT)
assert "extraheader" not in open(f"{REPO_DIR}/.git/config").read()     # the token was not saved

os.chdir(REPO_DIR)
sys.path[:0] = [f"{REPO_DIR}/src", f"{REPO_DIR}/scripts"]
import kaggle_stage1 as ks
from sira_cti.common import code_version, config_hash, load_config

CFG = load_config("configs/default.yaml")
the_hash = config_hash("configs/default.yaml")
print("commit:", head)
print("code version:", code_version(REPO_DIR))
print("config hash:", the_hash)
assert the_hash == EXPECTED_CONFIG_HASH, (
    f"config hash {the_hash} is not the expected {EXPECTED_CONFIG_HASH}: this commit's configs/default.yaml "
    "is not the one the Mac runs used.")

## 4. The corpus, the ontology files and the base index

From the attached dataset (see `docs/kaggle-run.md` for what to upload). They are copied into the checkout, where the config expects them. The base index is then asked two questions whose answers are known from the Mac.

In [ ]:
for line in ks.stage_inputs(INPUT_ROOT, REPO_DIR, CFG):
    print(line)
print()
for line in ks.check_base_index(CFG["index"]["base_dir"], total_docs=EXPECTED_TOTAL_DOCS, doc_freqs=EXPECTED_DOC_FREQS):
    print(line)
assert not code_version(REPO_DIR)["dirty"], "copying the data must not change anything git tracks"

## 5. Ollama: one server per GPU, then the model and its digest

Each server is started with `CUDA_VISIBLE_DEVICES` set to one card, so it cannot use the other. The **digest** is the fingerprint of the weights file. The name `qwen2.5:14b` is only a label; equal digests are what show Kaggle and the Mac run the same model.

In [ ]:
os.makedirs(f"{WORKING}/logs", exist_ok=True)
ACTIVE = PLAN if MODE == "full" else [{"gpu": 0, "port": PLAN[0]["port"]}]
SERVERS = [ks.start_ollama(e["gpu"], e["port"], f"{WORKING}/logs/ollama_gpu{e['gpu']}.log") for e in ACTIVE]
print(f"{len(SERVERS)} Ollama server(s) up:", [f"GPU {e['gpu']} -> port {e['port']}" for e in ACTIVE])

first = f"127.0.0.1:{ACTIVE[0]['port']}"
sh(f"OLLAMA_HOST={first} ollama pull {MODEL} > {WORKING}/logs/ollama_pull.log 2>&1")   # both servers share the files
sh(f"OLLAMA_HOST={first} ollama list")

from sira_cti.common import OllamaClient
for e in ACTIVE:
    client = OllamaClient(model=MODEL, host=f"http://127.0.0.1:{e['port']}")
    digest = client.model_digest()
    print(f"GPU {e['gpu']}: Ollama {client.server_version()}  {MODEL} digest {digest}")
    assert digest == EXPECTED_DIGEST, (
        f"DIGEST MISMATCH. Kaggle pulled {digest}, the Mac has {EXPECTED_DIGEST}. These are different weights "
        "under the same name, so this would not be the same model. Do not run. (docs/kaggle-run.md, 'If the digest differs')")
print("\nDigest matches the Mac's:", EXPECTED_DIGEST[:12])

## 6. Is the model on the GPU, not the CPU?

Each server loads the model and answers one tiny request. Ollama then reports how much of the model sits in GPU memory - it must be all of it. `nvidia-smi` should show about 10 GB in use on each card that is running.

In [ ]:
for e in ACTIVE:
    fraction = ks.warm_up(e["port"], MODEL, num_ctx=CFG["llm"].get("num_ctx"))
    print(f"GPU {e['gpu']} (port {e['port']}): {fraction:.0%} of the model is in GPU memory")
    assert fraction >= 0.999, (
        f"The server for GPU {e['gpu']} is running the model partly or wholly on the CPU ({fraction:.0%} on GPU). "
        f"See {WORKING}/logs/ollama_gpu{e['gpu']}.log")
for line in ks.gpu_report():
    print("  ", line)

## 7. Pick up an earlier session's output (full run only)

If the previous version of this notebook is attached as an input, its output files are copied into `/kaggle/working` and the run carries on from them. With nothing attached, the run starts from the beginning.

In [ ]:
def output_name(entry):
    return f"{RUN_NAME}.shard{entry['shard'].replace('/', 'of')}.jsonl" if entry.get("shard") else f"{RUN_NAME}.gpu{entry['gpu']}.jsonl"

if MODE == "full":
    for line in ks.restore_previous(INPUT_ROOT, WORKING, [output_name(e) for e in PLAN]):
        print(line)
    SHARES = ks.plan_shares(CFG["corpus"]["kb_dir"], PLAN, CFG["corpus"]["kinds"], ks.MAC_SECONDS_PER_DOC)
    print("\nEach GPU's share (hours are what the same share would take on the Mac, for comparing the two):")
    for s in SHARES:
        done = ks.count_lines(f"{WORKING}/{output_name(s)}")
        print(f"  GPU {s['gpu']}  --kinds {s['kinds']} --shard {s.get('shard')}: {s['documents']} documents "
              f"{s['documents_by_kind']}, {s['hours']:.1f} Mac-hours; {done} already done")
else:
    print("sample mode: nothing to resume")

## 8. Run

**Full:** two processes side by side, each told to stop 45 minutes before the session limit (less the time the cells above already used). Progress is printed every 10 minutes. This cell does not raise on a failure - whatever was written must still be saved when the notebook ends.

**Sample:** the 40 documents on GPU 0.

In [ ]:
env_for = lambda e: {"CUDA_VISIBLE_DEVICES": str(e["gpu"]), "JAVA_HOME": os.environ["JAVA_HOME"]}

if MODE == "full":
    limit = ks.stop_after_minutes(SESSION_LIMIT_HOURS, SAFETY_MINUTES, time.time() - SESSION_STARTED)
    print(f"{(time.time() - SESSION_STARTED) / 60:.0f} minutes used so far; each process will stop after {limit:g} minutes\n")
    JOBS = [{
        "label": f"GPU {s['gpu']} shard {s.get('shard')}",
        "cmd": ks.enrich_command(sys.executable, s, f"{WORKING}/{output_name(s)}", extra_args=FULL_ARGS, stop_after=limit),
        "output": f"{WORKING}/{output_name(s)}", "total": s["documents"],
        "log_path": f"{WORKING}/logs/enrich_gpu{s['gpu']}.log", "env": env_for(s),
    } for s in SHARES]
else:
    entry = ACTIVE[0]
    JOBS = [{
        "label": "sample on GPU 0",
        "cmd": ks.enrich_command(sys.executable, entry, f"{WORKING}/{SAMPLE_OUTPUT}", extra_args=SAMPLE_ARGS),
        "output": f"{WORKING}/{SAMPLE_OUTPUT}", "total": 40,
        "log_path": f"{WORKING}/logs/enrich_sample.log", "env": env_for(entry),
    }]

JOBS = ks.run_and_watch(JOBS, cwd=REPO_DIR, every_minutes=PROGRESS_EVERY_MINUTES if MODE == "full" else 2)

## 9. What is in `/kaggle/working`, and what to do next

In [ ]:
import json
for name in sorted(os.listdir(WORKING)):
    path = f"{WORKING}/{name}"
    if os.path.isfile(path):
        extra = f"  {ks.count_lines(path)} records" if name.endswith(".jsonl") else ""
        print(f"{os.path.getsize(path):>12,}  {name}{extra}")

for job in JOBS:
    manifest_file = job["output"] + ".manifest.json"
    if os.path.exists(manifest_file):
        m = json.load(open(manifest_file))
        print(f"\n{job['label']}: digest {str(m.get('model_digest'))[:12]}  shard {m.get('shard')}  "
              f"commit {m.get('code', {}).get('code_commit', '')[:12]}\n  machine {m.get('machine')}")

if MODE == "full":
    if all(job["complete"] for job in JOBS):
        merged = f"{WORKING}/{RUN_NAME}.jsonl"
        if os.path.exists(merged):
            print(f"\nALL DONE earlier: {merged} already exists.")
        else:
            total = sum(job["total"] for job in JOBS)
            result = sh([sys.executable, "scripts/merge_enrichment.py", *(job["output"] for job in JOBS),
                         "--output", merged, "--expect", str(total)], check=False)
            print("\nALL DONE. Download the merged file and its .manifest.json." if result.returncode == 0
                  else "\nBoth shards are complete but the merge was refused - see the message above.")
    else:
        left = sum(job["total"] - job["done"] for job in JOBS)
        print(f"\nNOT FINISHED: {left} documents still to do. Start another session with this version's "
              "output attached as an input (docs/kaggle-run.md, 'Chaining sessions').")

## 10. Sample mode: same replies as the Mac? And how long will the full run take here?

Runs only in sample mode. The comparison needs the Mac's file (`corpus_stratified_v4_14b_sections-v2.jsonl` and its `.manifest.json`) in the attached dataset; if it is not there, download the Kaggle file and run `scripts/compare_enrichment.py` on the Mac instead.

In [ ]:
if MODE == "sample" and JOBS[0]["done"]:
    sample = JOBS[0]["output"]
    references = [os.path.join(d, f) for d, _, files in os.walk(INPUT_ROOT) for f in files if f == MAC_REFERENCE]
    if references:
        os.makedirs("/tmp/mac-reference", exist_ok=True)
        for suffix in ("", ".manifest.json"):
            if os.path.exists(references[0] + suffix):
                shutil.copyfile(references[0] + suffix, f"/tmp/mac-reference/{MAC_REFERENCE}{suffix}")
        compare = [sys.executable, "scripts/compare_enrichment.py", f"/tmp/mac-reference/{MAC_REFERENCE}", sample]
        sh(compare, check=False)
        as_json = sh([*compare, "--json"], check=False, quiet=True).stdout
        open(f"{WORKING}/cross_hardware_comparison.json", "w").write(as_json)
    else:
        print(f"{MAC_REFERENCE} is not in the attached dataset - compare on the Mac (docs/kaggle-run.md).")

    measured = ks.seconds_per_doc_from(sample)
    print("\nMeasured here, seconds per document:", {k: round(v, 1) for k, v in measured.items()},
          " (Mac, sustained:", ks.MAC_SECONDS_PER_DOC, ")")
    usable = SESSION_LIMIT_HOURS - SAFETY_MINUTES / 60 - 0.4            # 0.4 h: installs and the model download
    for s in ks.plan_shares(CFG["corpus"]["kb_dir"], PLAN, CFG["corpus"]["kinds"], measured):
        print(f"  GPU {s['gpu']} shard {s.get('shard')}: {s['documents']} documents, about {s['hours']:.1f} hours "
              f"here = {s['hours'] / usable:.1f} sessions of {usable:.1f} usable hours")
    print("Read this as a rough guide: it rests on 10 documents of each kind, with the GPU to itself.")